# Frozen increasing-preference scoring

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
import hashlib,pickle
import numpy as np
from types import SimpleNamespace
def estimator_state_hash(model):
    """Hash full fitted member/scaler dictionaries, including non-array attributes."""
    return hashlib.sha256(pickle.dumps(
        [(m.__dict__, s.__dict__) for m, s in zip(model.models, model.scalers)],
        protocol=4)).hexdigest()

class FrozenPreferenceScore:
    """Mean class-0 support: CSV ranking 1 is LEFT > RIGHT, hence decrease."""
    def __init__(self, model):
        if not model.classifier or not model.preference or len(model.models) != 5 or len(model.scalers) != 5:
            raise ValueError('Requires exactly five frozen preference classifiers/scalers')
        self.model = model
        for member, scaler in zip(model.models, model.scalers):
            if list(member.classes_) != [0, 1] or member.coef_.shape != (1, 54) or scaler.n_features_in_ != 54:
                raise ValueError('Unexpected estimator schema or label order')
        self.initial_hash = estimator_state_hash(model)

    def assert_frozen(self):
        if estimator_state_hash(self.model) != self.initial_hash:
            raise RuntimeError('Frozen estimator/scaler state changed')

    def __call__(self, pair):
        self.assert_frozen()
        pair = np.asarray(pair, dtype=np.float64)
        if pair.shape != (54,) or not np.isfinite(pair).all():
            raise ValueError('Invalid raw preference pair; no sanitization permitted')
        support = []
        clipped_count = 0
        for member, scaler in zip(self.model.models, self.model.scalers):
            scaled = scaler.transform(pair.reshape(1, -1))
            if not np.isfinite(scaled).all():
                raise ValueError('Invalid scaled features')
            clipped_count += int(np.count_nonzero((scaled < 0) | (scaled > 1)))
            # Preserve native stored-scaler + clip preprocessing, log its support loss.
            support.append(float(member.predict_proba(np.clip(scaled, 0, 1))[0, 0]))
        q = float(np.mean(support))
        if not np.isfinite(q) or not 0 <= q <= 1:
            raise ValueError('Invalid estimator output')
        self.assert_frozen()
        return q, {'member_increase_support': support, 'clipped_member_features': clipped_count,
                   'member_feature_count': 270}

class PreferenceModel(FrozenPreferenceScore):
    def __init__(self,model_paths,scaler_paths):
        if len(model_paths)!=5 or len(scaler_paths)!=5:raise ValueError('Exactly five model/scaler paths required')
        def load(path):
            with open(path,'rb') as stream:return pickle.load(stream)
        model=SimpleNamespace(classifier=True,preference=True,models=[load(p) for p in model_paths],scalers=[load(p) for p in scaler_paths])
        if any(type(m).__name__!='LogisticRegression' for m in model.models) or any(type(s).__name__!='MinMaxScaler' for s in model.scalers):raise ValueError('Unexpected fitted artifact class')
        super().__init__(model)
        self.models,self.scalers=model.models,model.scalers
    def predict_q(self,features):
        rows=np.asarray(features,dtype=np.float64)
        if rows.ndim!=2 or rows.shape[1]!=54 or not np.isfinite(rows).all():raise ValueError('Expected finite N by 54 features')
        self.assert_frozen()
        result=np.mean([m.predict_proba(np.clip(s.transform(rows),0,1))[:,0] for m,s in zip(self.models,self.scalers)],axis=0)
        if not np.isfinite(result).all() or np.any((result<0)|(result>1)):raise ValueError('Invalid q')
        self.assert_frozen();return result

def load_preference_model():
    from contract import CODE,verify_inputs
    verify_inputs()
    folder=CODE/'affectively/models/solid/classifier/Linear'
    return PreferenceModel([folder/f'Cluster_0_classifier_preferences_linear_{i}.pkl' for i in range(5)], [folder/f'Cluster_0_classifier_preferences_linear_scaler_{i}.pkl' for i in range(5)])
print('Frozen increasing-preference scoring definitions/execution completed.')


Frozen increasing-preference scoring definitions/execution completed.


In [3]:
model=load_preference_model()
print('Runtime classes:',[type(m).__name__ for m in model.models])
print('Scaler classes:',[type(s).__name__ for s in model.scalers])
print('Features:',model.scalers[0].n_features_in_,'classes:',model.models[0].classes_.tolist())

Frozen runtime contract definitions/execution completed.


Runtime classes: ['LogisticRegression', 'LogisticRegression', 'LogisticRegression', 'LogisticRegression', 'LogisticRegression']
Scaler classes: ['MinMaxScaler', 'MinMaxScaler', 'MinMaxScaler', 'MinMaxScaler', 'MinMaxScaler']
Features: 54 classes: [0.0, 1.0]
